# Teste do Modelo 6: IBM Flowstate (granite-tsfm)

In [1]:
import pandas as pd
import torch
import numpy as np
import os

# --- 1. Configurações ---
DATA_DIR = "../../data"
HORIZONTE_PREVISAO = 14
CONTEXT_LENGTH = 96 # Comprimento do histórico, similar ao TimesFM

# --- 2. Carregar Dados ---
print("Carregando dados...")
hist_path = os.path.join(DATA_DIR, "hist.parquet")
df_hist = pd.read_parquet(hist_path)
print("Dados históricos carregados.")

# --- 3. Definir dispositivo ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsando dispositivo: {device}")

Carregando dados...
Dados históricos carregados.

Usando dispositivo: cpu


In [2]:
# ==============================================================================
# TESTE 6: IBM Flowstate (granite-tsfm)
# ==============================================================================
# API baseada nos notebooks oficiais: https://github.com/ibm-granite/granite-tsfm
# ==============================================================================

try:
    from transformers import AutoModelForPrediction
    from torch.utils.data import TensorDataset, DataLoader

    print("\n--- Testando IBM Flowstate ---")
    
    # Garantir que temos dados suficientes
    if len(df_hist) < CONTEXT_LENGTH:
        raise ValueError(f"O modelo requer um histórico de pelo menos {CONTEXT_LENGTH} pontos.")

    # 1. Carregar o modelo pré-treinado (usando TinyTimeMixer como exemplo)
    model = AutoModelForPrediction.from_pretrained(
        "ibm-granite/granite-7b-ttm",
        device_map=device,
        trust_remote_code=True
    )

    # 2. Preparar dados
    # O modelo espera um batch de dados. Para nosso caso, será um batch de 1.
    # Pegamos os últimos `CONTEXT_LENGTH` pontos do nosso histórico.
    context_data = df_hist['target'].values[-CONTEXT_LENGTH:]
    
    # O modelo espera o formato [batch_size, sequence_length, num_variates]
    # Para nosso caso: [1, CONTEXT_LENGTH, 1]
    input_tensor = torch.tensor(context_data, dtype=torch.float32).unsqueeze(0).unsqueeze(-1).to(device)
    
    # Criar um DataLoader
    dataset = TensorDataset(input_tensor)
    dataloader = DataLoader(dataset, batch_size=1)
    batch = next(iter(dataloader))
    
    # 3. Rodar a previsão
    print(f"Rodando previsão para {HORIZONTE_PREVISAO} passos...")
    forecast = model.predict(batch, prediction_length=HORIZONTE_PREVISAO)
    
    # A saída é um tensor com o formato [batch_size, prediction_length, num_variates]
    forecast_values = forecast[0, :, 0].cpu().numpy()

    print("Previsão (primeiros 5 valores):", forecast_values.round(2)[:5])
    print("Teste do Flowstate (granite-tsfm) concluído.\n")

except ImportError:
    print("granite-tsfm não instalado. Pulando teste.")
except Exception as e:
    print(f"Erro ao rodar Flowstate: {e}")

granite-tsfm não instalado. Pulando teste.
